# Tome 8, Chapter 9: The Hyperbolic Plane and the Modular Group (Python companion)

Cutting sequences as continued fractions, trees embedded in the hyperbolic plane, and lattice reduction as the walk into the fundamental domain.

*Before each solution, predict what the output will look like.*

## Exercises

**P9.1 (★) Cutting sequences and continued fractions.** (a) Follow the h-line from $y\in(-1,0)$ to $x>1$ through the Farey tessellation: the Farey triangle it enters after crossing an edge $(\frac ab,\frac cd)$ has third vertex the mediant $\frac{a+c}{b+d}$, and the line leaves through the side towards $x$. Record the letters $L$, $R$ (Theorem 9.8) and compare with the continued fraction of $x$ for $x=\pi$, $e$, $1+\sqrt2$ and the golden ratio. (b) Check along the way that the mediants visited at the turns are the convergents $\frac{p_n}{q_n}$ of Tome 1.

**P9.2 (★★) CS application: embedding trees in the hyperbolic plane.** Hierarchies (taxonomies, file systems, the WordNet noun hierarchy) are trees, and machine learning embeds them in the hyperbolic plane because a tree, like the hyperbolic plane, grows exponentially (Proposition 9.3: a circle of radius $r$ has length $2\pi\sinh r$). (a) Implement Sarkar's construction (2011) in the disc model: put the root at $0$, and place the children of each node at hyperbolic distance $\tau$ from it, at equally spaced angles around it, the parent occupying one of the slots. Measure the distortion of the binary tree of depth $6$: the range of $d_{\mathbb H}(u,v)/(\tau\, d_{\text{tree}}(u,v))$ over all pairs. (b) Why does the distortion tend to $1$ as $\tau$ grows? (c) Repeat in double precision for deeper trees and watch it fail: how many bits does a node at depth $k$ need?

**P9.3 (★★) CS application: reducing a lattice basis is moving a point into $F$.** (a) Implement the Lagrange–Gauss reduction of a basis $(b_1,b_2)$ of a lattice in $\R^2$: while $|b_2|<|b_1|$ swap them, and subtract from $b_2$ the nearest integer multiple of $b_1$. Show that, with $\tau=b_2/b_1$ (as complex numbers, after orienting the basis), the two moves are $S$ and a power of $T$, and that the algorithm stops exactly when $\tau\in F$. (b) Count its steps on random bases with entries of $40$ digits, and compare with the number of steps of Euclid's algorithm. (c) Use reduced forms (Exercise 9.4) to list the discriminants $-D$, $D\le200$, of class number one; you should find Gauss's list, proved complete by Heegner, Baker and Stark.

## Solutions

### P9.1

In [ ]:
from fractions import Fraction
import mpmath
mpmath.mp.dps = 60

def cutting_sequence(x, n_letters=40):
    """Walk from the edge (0, oo): the current edge joins a/b (left vertex, real) and c/d (right vertex, possibly oo=1/0)."""
    a, b, c, d = 0, 1, 1, 0            # the edge from 0/1 to 1/0, crossed by the line from y in (-1,0) to x > 1
    letters, turns = [], []
    for _ in range(n_letters):
        m_num, m_den = a + c, b + d    # third vertex of the next triangle: the mediant
        if x > mpmath.mpf(m_num) / m_den:
            a, b = m_num, m_den        # leave through (m, c/d): the kept vertex c/d lies beyond x, on the left
            letters.append("L")
        else:
            c, d = m_num, m_den        # leave through (a/b, m): the kept vertex a/b lies in (y, x), on the right
            letters.append("R")
        if len(letters) > 1 and letters[-1] != letters[-2]:
            turns.append(Fraction(a, b) if letters[-1] == "R" else Fraction(c, d))
    return "".join(letters), turns

def runs(word):
    out, k = [], 1
    for i in range(1, len(word)):
        if word[i] == word[i - 1]: k += 1
        else: out.append(k); k = 1
    return out

def cf(x, n=8):
    out = []
    for _ in range(n):
        a = int(mpmath.floor(x)); out.append(a); x = 1 / (x - a)
    return out

for name, x in [("pi", mpmath.pi), ("e", mpmath.e), ("1+sqrt2", 1 + mpmath.sqrt(2)), ("golden", (1 + mpmath.sqrt(5)) / 2)]:
    w, turns = cutting_sequence(x, 60)
    print(f"{name:8s} {w[:40]}...")
    print(f"         run lengths {runs(w)[:7]}   continued fraction {cf(x, 7)}")

The run lengths of the cutting sequence are the partial quotients: $\pi=[3;7,15,1,292,\dots]$ starts with $LLL$, then seven $R$, then fifteen $L$ (the $292$ would need far more letters), $e=[2;1,2,1,1,4,1,\dots]$, and the quadratic irrationals give periodic words, $LLRR$ repeated for $1+\sqrt2$ and $LR$ for the golden ratio. The walk is the Stern–Brocot (or Farey) descent towards $x$: each letter replaces one end of the current interval by the mediant.

In [ ]:
w, turns = cutting_sequence(mpmath.pi, 40)
print("vertices where the cutting sequence turns:", [str(t) for t in turns])
p0, q0, p1, q1 = 1, 0, 3, 1
conv = ["3"]
for a in [7, 15, 1]:
    p0, q0, p1, q1 = p1, q1, a * p1 + p0, a * q1 + q0
    conv.append(f"{p1}/{q1}")
print("convergents of pi:                    ", conv)

At each change of letter the vertex that has been shared by the whole previous run is a convergent of $x$: $\frac31$, $\frac{22}7$, $\frac{333}{106}$, $\frac{355}{113}$. The triangles between two turns form a fan around that convergent, and the number of triangles in the fan is the next partial quotient. The best rational approximations are the points of the boundary that the line passes closest to, in the hyperbolic sense.

### P9.2

In [ ]:
import numpy as np
import itertools

def mobius_to_origin(z0):            # disc isometry sending z0 to 0
    return lambda w: (w - z0) / (1 - np.conj(z0) * w)
def mobius_from_origin(z0):          # its inverse
    return lambda w: (w + z0) / (1 + np.conj(z0) * w)

def dist(z, w, lib=np):
    num = 2 * abs(z - w) ** 2
    den = (1 - abs(z) ** 2) * (1 - abs(w) ** 2)
    return lib.acosh(1 + num / den)

def sarkar_binary_tree(depth, tau, mp=False):
    """Nodes as (position, parent index, depth); children of a node spread at equal angles, parent in one slot."""
    if mp:
        r = mpmath.tanh(mpmath.mpf(tau) / 2); expi = lambda a: mpmath.expj(a); conj = mpmath.conj
    else:
        r = np.tanh(tau / 2); expi = lambda a: np.exp(1j * a); conj = np.conj
    nodes = [(0 * r, -1, 0)]
    frontier = [0]
    for dpt in range(depth):
        new = []
        for i in frontier:
            z, par, _ = nodes[i]
            to0 = lambda w, z=z: (w - z) / (1 - conj(z) * w)
            back = lambda w, z=z: (w + z) / (1 + conj(z) * w)
            if par < 0:
                angles = [0, np.pi]
            else:
                p = to0(nodes[par][0])
                a0 = mpmath.arg(p) if mp else np.angle(p)
                angles = [a0 + 2 * np.pi / 3, a0 + 4 * np.pi / 3]
            for a in angles:
                nodes.append((back(r * expi(a)), i, dpt + 1)); new.append(len(nodes) - 1)
        frontier = new
    return nodes

def tree_dist(nodes, i, j):
    di, dj = nodes[i][2], nodes[j][2]; d = 0
    while i != j:
        if di >= dj: i = nodes[i][1]; di -= 1
        else: j = nodes[j][1]; dj -= 1
        d += 1
    return d

for tau in (1.0, 2.0, 4.0, 8.0):
    nodes = sarkar_binary_tree(6, tau, mp=True)
    ratios = [float(dist(nodes[i][0], nodes[j][0], mpmath)) / (tau * tree_dist(nodes, i, j))
              for i, j in itertools.combinations(range(len(nodes)), 2)]
    print(f"tau = {tau:4.1f}: {len(nodes)} nodes, d_H / (tau d_tree) in [{min(ratios):.4f}, {max(ratios):.4f}]")

Every ratio is at most $1$: the path through the tree has hyperbolic length $\tau\,d_{\text{tree}}$, and the geodesic is shorter. The lower end of the range rises towards $1$ as $\tau$ grows. The reason is the thinness of hyperbolic triangles: at a node, the incoming and outgoing edges make an angle of at least $\frac{2\pi}3$, and a path of long geodesic segments meeting at angles bounded away from $0$ is almost a geodesic, losing only a bounded amount, of order $\log\frac1{\sin(\theta/2)}$, at each corner. Relative to the total length $\tau\,d_{\text{tree}}$ that loss shrinks as $\tau$ grows. In the Euclidean plane no such embedding exists: the $2^d$ leaves of a binary tree of depth $d$ are pairwise at tree distance at least $2$ but all within distance $d$ of the root, and a disc of radius proportional to $d$ cannot hold $2^d$ well-separated points, so the distortion must grow at least like $2^{d/2}/d$.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
for depth, tau in [(6, 4.0), (8, 4.0), (10, 4.0), (12, 4.0)]:
    leaf = sarkar_binary_tree(depth, tau, mp=False)[-1][0]
    leaf_mp = sarkar_binary_tree(depth, tau, mp=True)[-1][0]
    print(f"depth {depth:2d}: 1-|z| = {float(1 - abs(leaf_mp)):.3e};  d(root, leaf) in double precision {dist(0j, leaf):8.3f},"
          f" with 60 digits {float(dist(0, leaf_mp, mpmath)):8.3f}")

A node at depth $k$ sits at hyperbolic distance about $k\tau$ from the origin, that is at Euclidean distance $1-|z|\approx2e^{-k\tau}$ from the boundary circle. In double precision, $1-|z|$ is only resolved down to about $10^{-16}$: the computed distances drift away from the correct ones as soon as $1-|z|$ approaches that scale, and beyond $k\tau\approx37$ the leaf is rounded onto the circle, where distances are infinite or meaningless. Representing a node at depth $k$ needs about $k\tau\log_2e$ bits: this is the precision–distortion trade-off analysed by Sala, De Sa, Gu and Ré (2018), and the reason practical hyperbolic embeddings use moderate $\tau$ or other models of the plane.

### P9.3

In [ ]:
import random

def lagrange_gauss(b1, b2):
    steps = 0
    while True:
        n1 = b1[0] * b1[0] + b1[1] * b1[1]
        mu = (b1[0] * b2[0] + b1[1] * b2[1])
        k = (2 * mu + n1) // (2 * n1)                         # nearest integer to mu / n1, exactly
        if k:
            b2 = (b2[0] - k * b1[0], b2[1] - k * b1[1]); steps += 1      # tau -> tau - k: the move T^-k
        if b2[0] ** 2 + b2[1] ** 2 < n1:
            b1, b2 = b2, b1; steps += 1                       # tau -> 1/tau up to sign: the move S
        else:
            return b1, b2, steps

def euclid_steps(a, b):
    s = 0
    while b: a, b = b, a % b; s += 1
    return s

from math import gcd
def unimodular(N):                   # a random matrix of SL_2(Z) with entries of size about N
    while True:
        p, q = random.randrange(N), random.randrange(N)
        if gcd(p, q) == 1: break
    # solve p*s - q*r = 1 with the extended Euclidean algorithm
    def egcd(a, b):
        if b == 0: return a, 1, 0
        g, x, y = egcd(b, a % b); return g, y, x - (a // b) * y
    g, x, y = egcd(p, q)             # p*x + q*y = 1
    return p, q, -y, x               # p*x - q*(-y) = 1

random.seed(9)
N = 10 ** 40
e1, e2 = (3, 1), (1, 4)              # a basis of some lattice
tot_g, tot_e = 0, 0
for _ in range(300):
    p, q, r, s = unimodular(N)
    assert p * s - q * r == 1
    b1 = (p * e1[0] + q * e2[0], p * e1[1] + q * e2[1])   # the same lattice, in a terrible basis
    b2 = (r * e1[0] + s * e2[0], r * e1[1] + s * e2[1])
    r1, r2, steps = lagrange_gauss(b1, b2); tot_g += steps
    tot_e += euclid_steps(p, q)
print("a reduced basis found from the last terrible one:", r1, r2)
print(f"average moves: Lagrange-Gauss {tot_g / 300:.1f}, Euclid on the coefficients {tot_e / 300:.1f}")

Write $b_1$, $b_2$ as complex numbers, with $\operatorname{Im}(b_2/b_1)>0$ (swap if needed); the lattice up to rotation and scaling is the point $\tau=b_2/b_1\in\mathbb H$. Subtracting $k b_1$ from $b_2$ is $\tau\mapsto\tau-k$, the move $T^{-k}$, and choosing $k$ nearest to $\operatorname{Re}\tau$ puts $|\operatorname{Re}\tau|\le\frac12$. Swapping the vectors is $\tau\mapsto1/\tau$, which becomes $-1/\tau$, the move $S$, after re-orienting the basis. The algorithm stops when $|b_2|\ge|b_1|$, that is $|\tau|\ge1$, with $|\operatorname{Re}\tau|\le\frac12$: exactly $\tau\in F$, and $b_1$ is then a shortest vector of the lattice. It is Euclid's algorithm in disguise: starting from a basis scrambled by a unimodular matrix with $40$-digit entries, it needs a number of moves (translations and swaps counted separately) comparable to the number of division steps of Euclid's algorithm on those entries, logarithmic in their size. In higher dimension its descendant is the LLL algorithm of Lenstra, Lenstra and Lovász (1982), the workhorse of lattice cryptanalysis met in the computing track.

In [ ]:
def reduced_forms(D):                    # forms ax^2+bxy+cy^2 with b^2-4ac = -D, reduced in Gauss's sense
    out = []
    a = 1
    while 3 * a * a <= D:
        for b in range(-a + 1, a + 1):
            if (b * b + D) % (4 * a) == 0:
                c = (b * b + D) // (4 * a)
                if c >= a and not (c == a and b < 0):
                    out.append((a, b, c))
        a += 1
    return out

one = [D for D in range(3, 201) if D % 4 in (0, 3) and len(reduced_forms(D)) == 1]
print("discriminants -D with class number one, D <= 200:", one)
print("h(-20) =", len(reduced_forms(20)), reduced_forms(20), "  h(-23) =", len(reduced_forms(23)), reduced_forms(23))

The list is Gauss's nine: $-3,-4,-7,-8,-11,-19,-43,-67,-163$. (We counted all reduced forms. Counting only primitive ones, with $\gcd(a,b,c)=1$, would add $-12$, $-16$, $-27$, $-28$, which have one primitive class and one imprimitive form such as $2x^2+2xy+2y^2$; these discriminants belong to non-maximal orders.) Gauss conjectured in 1801 that the list of imaginary quadratic fields with class number one is complete; Heegner (1952), Baker (1966) and Stark (1967) proved it. The last, $163$, is behind the near-integer $e^{\pi\sqrt{163}}$ of Tome 12.